# Build source-holdout splits (mentor protocol)

Canonical builder for `data/splits/source_holdout/`.

| Rule | Behavior |
|------|----------|
| Sources | **Raw** — Simsonsun Dataset_2, MPDD, PI jsonl, Bravansky compact CSV |
| Hold out | Entire source (attacks + native benign). PI fold never trains on PI-benign |
| Test | Exactly **200** held-out attacks + **200** shared benign (same on every four-source fold) |
| Train attacks | Exactly **200 from each** other source |
| Fit benign | Exactly **50** every fold (PI fold: ChatGPT only) |
| Val | 20% **within each** source/benign piece |
| Sizes | All four-source folds identical (`520 / 130 / 400`) |
| Fifth set | Neuralchemy `200+200` test; fit uses 200 from each of four sources |

Run all cells top to bottom. Optional CLI: `python scripts/build_source_holdout.py` (executes this notebook's code).


## 0. Imports and paths


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)
from sklearn.model_selection import train_test_split

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import EXTERNAL_TEST_MAIN_CSV, RANDOM_SEED, SOURCE_HOLDOUT_DIR
from src.splits.source_holdout import (
    ATTACK_SOURCES,
    EXTERNAL_SOURCE,
    SOURCE_BENIGN,
    VAL_FRACTION,
    assign_exclusive_source,
    drop_near_duplicates,
    load_bravansky_raw,
    load_chatgpt_benign,
    load_mpdd,
    load_prompt_injection,
    load_simsonsun,
    write_fold,
    _norm,
)

OUT = SOURCE_HOLDOUT_DIR
print("ROOT:", ROOT)
print("OUT :", OUT)


## 1. Knobs

Change these to resize the matched holdout.  
`N_ATTACK_TRAIN_PER_SOURCE` / `N_ATTACK_TEST` cannot exceed the smallest cleaned source (~249 PI).  
`N_BENIGN_FIT` must fit in ChatGPT alone (≤51) so the PI fold stays size-matched.


In [ ]:
N_ATTACK_TEST = 200
N_BENIGN_TEST = 200
N_ATTACK_TRAIN_PER_SOURCE = 200
N_NEURALCHEMY_ATTACK = 200
N_NEURALCHEMY_BENIGN = 200
N_BENIGN_FIT = 50
BRAVANSKY_RAW_CAP = 2500
SEED = RANDOM_SEED

assert N_ATTACK_TEST > 0 and N_BENIGN_TEST > 0
assert N_ATTACK_TRAIN_PER_SOURCE > 0
assert N_BENIGN_FIT > 0
print(
    f"Uniform: test={N_ATTACK_TEST} inj + {N_BENIGN_TEST} shared ben; "
    f"train={N_ATTACK_TRAIN_PER_SOURCE}/source; fit_benign={N_BENIGN_FIT}"
)


## 2. Helpers


In [ ]:
def sample_n(frame: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(frame) < n:
        raise ValueError(
            f"Need {n} rows, have {len(frame)} "
            f"(sources={frame['source'].unique().tolist()})"
        )
    return frame.sample(n=n, random_state=seed).reset_index(drop=True)


def fit_split_by_source(pieces: list[pd.DataFrame]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """80/20 within each piece so every source keeps the same train/val ratio."""
    trains, vals = [], []
    for i, piece in enumerate(pieces):
        if len(piece) < 5:
            raise ValueError(f"Piece {i} too small to split: {len(piece)}")
        strat = piece["label"] if piece["label"].nunique() > 1 else None
        tr, va = train_test_split(
            piece,
            test_size=VAL_FRACTION,
            random_state=SEED + i,
            stratify=strat,
        )
        trains.append(tr)
        vals.append(va)
    train = pd.concat(trains, ignore_index=True)
    val = pd.concat(vals, ignore_index=True)
    return train.reset_index(drop=True), val.reset_index(drop=True)


## 3. Load raw sources


In [ ]:
simsonsun = load_simsonsun()
mpdd = load_mpdd()
pi_attacks, pi_benign = load_prompt_injection()
chatgpt = load_chatgpt_benign()
bravansky = load_bravansky_raw(max_unique=BRAVANSKY_RAW_CAP)

inventory = pd.DataFrame(
    [
        {"source": "simsonsun", "role": "attack", "n": len(simsonsun)},
        {"source": "mpdd", "role": "attack", "n": len(mpdd)},
        {"source": "prompt_injection", "role": "attack", "n": len(pi_attacks)},
        {"source": "bravansky_raw", "role": "attack", "n": len(bravansky)},
        {"source": "prompt_injection_benign", "role": "benign", "n": len(pi_benign)},
        {"source": "chatgpt_benign", "role": "benign", "n": len(chatgpt)},
    ]
)
display(inventory)


## 4. Exact exclusivity + MiniLM near-duplicates


In [ ]:
attacks, taken, exact_overlaps = assign_exclusive_source(
    [
        ("simsonsun", simsonsun),
        ("bravansky", bravansky),
        ("mpdd", mpdd),
        ("prompt_injection", pi_attacks),
    ]
)
attacks, near_dup_report = drop_near_duplicates(attacks)

taken = set()
for frame in attacks.values():
    taken.update(frame["prompt_key"].tolist())

after = pd.DataFrame(
    [{"source": name, "n_attacks": len(attacks[name])} for name in ATTACK_SOURCES]
)
display(after)
print("Exact overlaps removed:", exact_overlaps)
print(
    "Near-duplicates removed:",
    near_dup_report["removed"],
    near_dup_report["removed_by_source"],
)
need = max(N_ATTACK_TEST, N_ATTACK_TRAIN_PER_SOURCE)
assert int(after["n_attacks"].min()) >= need, (
    f"Need ≥{need} per source after cleaning; got\n{after}"
)


## 5. Shared benign (test-only) + fit reservoir

Shared **200** = PI-benign only, so ChatGPT can fill the PI fold's train benign and every four-source fold stays the same size.


In [ ]:
benign_all = pd.concat([chatgpt, pi_benign], ignore_index=True)
benign_all = benign_all.drop_duplicates(subset=["prompt_key"]).reset_index(drop=True)
benign_all = benign_all[~benign_all["prompt_key"].isin(taken)].reset_index(drop=True)

pi_ben_pool = benign_all[benign_all["source"] == "prompt_injection_benign"]
chatgpt_pool = benign_all[benign_all["source"] == "chatgpt_benign"]
assert len(pi_ben_pool) >= N_BENIGN_TEST, (
    f"Need ≥{N_BENIGN_TEST} PI-benign for shared test; have {len(pi_ben_pool)}"
)
assert len(chatgpt_pool) >= N_BENIGN_FIT, (
    f"Need ≥{N_BENIGN_FIT} ChatGPT benign for PI-fold train; have {len(chatgpt_pool)}"
)

shared_benign = sample_n(pi_ben_pool, N_BENIGN_TEST, SEED)
shared_keys = set(shared_benign["prompt_key"])
leftover_pi = pi_ben_pool[~pi_ben_pool["prompt_key"].isin(shared_keys)]
benign_fit_master = pd.concat([leftover_pi, chatgpt_pool], ignore_index=True)

print("Shared benign test:", len(shared_benign), shared_benign["source"].value_counts().to_dict())
print(
    "Benign fit master:",
    len(benign_fit_master),
    benign_fit_master["source"].value_counts().to_dict(),
)
print(
    f"Leftover PI-benign after shared ({len(leftover_pi)}) stay out of the PI fold "
    "(held out of train; not added to test)."
)


## 6. Fold builders


In [ ]:
def fit_benign_for_fold(held_out: str, seed: int) -> pd.DataFrame:
    banned = SOURCE_BENIGN.get(held_out, ())
    allowed = benign_fit_master[~benign_fit_master["source"].isin(banned)]
    return sample_n(allowed, N_BENIGN_FIT, seed)


def sample_train_sources(held_out: str | None, seed_base: int) -> list[pd.DataFrame]:
    pieces = []
    for i, name in enumerate(ATTACK_SOURCES):
        if name == held_out:
            continue
        pieces.append(sample_n(attacks[name], N_ATTACK_TRAIN_PER_SOURCE, seed_base + i))
    return pieces


def build_four_source_fold(held_out: str):
    test_attacks = sample_n(
        attacks[held_out], N_ATTACK_TEST, SEED + hash(held_out) % 10_000
    )
    test_keys = set(test_attacks["prompt_key"])

    attack_pieces = []
    for i, name in enumerate(ATTACK_SOURCES):
        if name == held_out:
            continue
        pool = attacks[name]
        pool = pool[~pool["prompt_key"].isin(test_keys)]
        attack_pieces.append(sample_n(pool, N_ATTACK_TRAIN_PER_SOURCE, SEED + 1000 + i))

    fit_benign = fit_benign_for_fold(held_out, SEED + 50 + hash(held_out) % 1000)
    assert len(fit_benign) == N_BENIGN_FIT
    if held_out == "prompt_injection":
        assert set(fit_benign["source"]) == {"chatgpt_benign"}

    train, val = fit_split_by_source(attack_pieces + [fit_benign])
    test = pd.concat([test_attacks, shared_benign], ignore_index=True)

    seen = {_norm(p) for p in list(train["prompt"]) + list(val["prompt"])}
    leak = sum(_norm(p) in seen for p in test["prompt"])
    assert leak == 0, f"{held_out}: {leak} test prompts leak into train/val"
    assert (test["label"] == "injection").sum() == N_ATTACK_TEST
    assert (test["label"] == "benign").sum() == N_BENIGN_TEST

    fit_inj = pd.concat(attack_pieces, ignore_index=True)
    counts = fit_inj["source"].value_counts().to_dict()
    assert all(v == N_ATTACK_TRAIN_PER_SOURCE for v in counts.values()), counts
    assert len(counts) == 3, counts
    assert shared_keys.issubset(set(test.loc[test["label"] == "benign", "prompt_key"]))
    return train, val, test, counts


def build_neuralchemy_fold():
    neuralchemy = pd.read_csv(EXTERNAL_TEST_MAIN_CSV)
    neuralchemy["prompt_key"] = neuralchemy["prompt"].map(_norm)
    dropped = int(neuralchemy["prompt_key"].isin(taken).sum())
    neuralchemy = neuralchemy[~neuralchemy["prompt_key"].isin(taken)].copy()
    neuralchemy["source"] = "neuralchemy:" + neuralchemy["source"].astype(str)
    inj = neuralchemy[neuralchemy["label"] == "injection"]
    ben = neuralchemy[neuralchemy["label"] == "benign"]
    test = pd.concat(
        [
            sample_n(inj, N_NEURALCHEMY_ATTACK, SEED + 7),
            sample_n(ben, N_NEURALCHEMY_BENIGN, SEED + 11),
        ],
        ignore_index=True,
    )
    attack_pieces = sample_train_sources(None, SEED + 2000)
    fit_benign = sample_n(benign_fit_master, N_BENIGN_FIT, SEED + 99)
    train, val = fit_split_by_source(attack_pieces + [fit_benign])
    seen = {_norm(p) for p in list(train["prompt"]) + list(val["prompt"])}
    assert sum(_norm(p) in seen for p in test["prompt"]) == 0
    counts = pd.concat(attack_pieces, ignore_index=True)["source"].value_counts().to_dict()
    assert all(v == N_ATTACK_TRAIN_PER_SOURCE for v in counts.values()), counts
    return train, val, test, dropped, counts

print("Builders ready.")


## 7. Build all folds and write CSVs


In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
summary_rows = []
fold_payload = {}

for held in ATTACK_SOURCES:
    train, val, test, train_atk_counts = build_four_source_fold(held)
    write_fold(OUT / held, train, val, test)
    fold_payload[held] = {
        "train": len(train),
        "val": len(val),
        "test": len(test),
        "test_injection": int((test["label"] == "injection").sum()),
        "test_benign": int((test["label"] == "benign").sum()),
        "test_shared_benign": int(len(shared_benign)),
        "fit_attack_per_source": train_atk_counts,
        "train_injection_by_source": train.loc[train["label"] == "injection", "source"]
        .value_counts()
        .to_dict(),
        "val_injection_by_source": val.loc[val["label"] == "injection", "source"]
        .value_counts()
        .to_dict(),
        "train_benign_by_source": train.loc[train["label"] == "benign", "source"]
        .value_counts()
        .to_dict(),
        "val_benign_by_source": val.loc[val["label"] == "benign", "source"]
        .value_counts()
        .to_dict(),
    }
    summary_rows.append(
        {
            "fold": held,
            "train": len(train),
            "val": len(val),
            "test": len(test),
            "test_injection": fold_payload[held]["test_injection"],
            "test_benign": fold_payload[held]["test_benign"],
            "test_shared_benign": len(shared_benign),
        }
    )
    print(
        f"{held:18s} train={len(train):4d} val={len(val):3d} test={len(test):3d} "
        f"(inj={fold_payload[held]['test_injection']}, ben={fold_payload[held]['test_benign']})"
    )
    print(f"{'':18s} fit_attacks/source={train_atk_counts}")
    print(f"{'':18s} train_benign={fold_payload[held]['train_benign_by_source']}")

train, val, test, dropped, train_atk_counts = build_neuralchemy_fold()
write_fold(OUT / EXTERNAL_SOURCE, train, val, test)
fold_payload[EXTERNAL_SOURCE] = {
    "train": len(train),
    "val": len(val),
    "test": len(test),
    "test_injection": int((test["label"] == "injection").sum()),
    "test_benign": int((test["label"] == "benign").sum()),
    "dropped_overlap_with_four_sources": dropped,
    "fit_attack_per_source": train_atk_counts,
    "train_injection_by_source": train.loc[train["label"] == "injection", "source"]
    .value_counts()
    .to_dict(),
    "val_injection_by_source": val.loc[val["label"] == "injection", "source"]
    .value_counts()
    .to_dict(),
    "train_benign_by_source": train.loc[train["label"] == "benign", "source"]
    .value_counts()
    .to_dict(),
    "val_benign_by_source": val.loc[val["label"] == "benign", "source"]
    .value_counts()
    .to_dict(),
}
summary_rows.append(
    {
        "fold": EXTERNAL_SOURCE,
        "train": len(train),
        "val": len(val),
        "test": len(test),
        "test_injection": fold_payload[EXTERNAL_SOURCE]["test_injection"],
        "test_benign": fold_payload[EXTERNAL_SOURCE]["test_benign"],
        "test_shared_benign": "",
    }
)
print(
    f"{EXTERNAL_SOURCE:18s} train={len(train):4d} val={len(val):3d} test={len(test):3d} "
    f"(inj={fold_payload[EXTERNAL_SOURCE]['test_injection']}, "
    f"ben={fold_payload[EXTERNAL_SOURCE]['test_benign']})"
)


## 8. Uniformity check + figure


In [ ]:
summary = pd.DataFrame(summary_rows)
four = summary[summary["fold"].isin(ATTACK_SOURCES)]
assert four["test_injection"].nunique() == 1
assert four["test_benign"].nunique() == 1
assert four["test"].nunique() == 1
assert four["train"].nunique() == 1, four["train"].tolist()
assert four["val"].nunique() == 1, four["val"].tolist()
print("Four-source folds are fully size-uniform.")
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
axes[0].bar(summary["fold"], summary["test_injection"], color="#c44e52")
axes[0].set_title("Test injections (uniform 200)")
axes[0].tick_params(axis="x", rotation=30)
axes[1].bar(summary["fold"], summary["test_benign"], color="#4c72b0")
axes[1].set_title("Test benign (uniform shared 200)")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
fig_path = OUT / "holdout_test_sizes.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print("Saved", fig_path)
plt.show()


## 9. Manifest + train-benign inventory


In [ ]:
benign_docs = []
for fold, payload in fold_payload.items():
    row = {"fold": fold}
    for k, v in payload.get("train_benign_by_source", {}).items():
        row[f"train_{k}"] = v
    benign_docs.append(row)
benign_table = pd.DataFrame(benign_docs).fillna(0)
benign_table.to_csv(OUT / "train_benign_by_fold.csv", index=False)
display(benign_table)

manifest = {
    "seed": SEED,
    "protocol": "final mentor source-holdout (fully uniform + raw)",
    "output_dir": "data/splits/source_holdout",
    "built_by": "notebooks/build_source_holdout.ipynb",
    "rules": [
        "Raw sources only (Simsonsun Dataset_2, MPDD, PI jsonl, Bravanskycompact CSV)",
        "Leave-one-source-out: held-out attacks never in train/val",
        "Held-out source native benign never in that fold's train/val",
        "Four-source folds have identical train/val/test sizes (PI included)",
        "Shared 200 benign = PI-benign only; identical on every four-source test",
        f"Exactly {N_ATTACK_TRAIN_PER_SOURCE} train-pool attacks from EACH other source",
        f"Exactly {N_BENIGN_FIT} fit benign every fold (PI fold: ChatGPT only)",
        "Leftover PI-benign not in shared stay out of the PI fold entirely (not on test)",
        "Val = 20% split within each source piece",
        "Method A fixed paper threshold; tune Method B / fold baselines on val only",
    ],
    "knobs": {
        "n_attack_test": N_ATTACK_TEST,
        "n_shared_benign_test": N_BENIGN_TEST,
        "n_attack_train_per_source": N_ATTACK_TRAIN_PER_SOURCE,
        "n_benign_fit": N_BENIGN_FIT,
        "bravansky_raw_cap": BRAVANSKY_RAW_CAP,
        "neuralchemy_attack": N_NEURALCHEMY_ATTACK,
        "neuralchemy_benign": N_NEURALCHEMY_BENIGN,
    },
    "near_duplicate_filter": near_dup_report,
    "exact_overlaps": exact_overlaps,
    "shared_benign_test": {
        "rows": int(len(shared_benign)),
        "by_source": shared_benign["source"].value_counts().to_dict(),
        "note": "All from prompt_injection_benign so ChatGPT can fill PI-fold train benign.",
    },
    "pi_benign_held_out_unused": int(len(leftover_pi)),
    "folds": fold_payload,
    "how_to_score": [
        "python scripts/eval_loo_holdout.py",
        "results land in results/source_holdout/",
    ],
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
(OUT / "near_duplicate_spot_check.json").write_text(
    json.dumps(near_dup_report.get("spot_check", []), indent=2) + "\n", encoding="utf-8"
)
summary.to_csv(OUT / "summary_table.csv", index=False)
print("Wrote", OUT / "manifest.json")
print("Wrote", OUT / "summary_table.csv")
print("Wrote", OUT / "train_benign_by_fold.csv")


## 10. Peek at one fold


In [ ]:
example = pd.read_csv(OUT / "mpdd" / "test.csv")
print(example["label"].value_counts().to_dict())
print(example["source"].value_counts().to_dict())
display(example.head(3))
display(example.tail(3))
